# Sampling Strategy Comparison for Contact Prediction

Compare beam search, nucleus (top-p) sampling, temperature scaling, and top-k sampling
for generating protein contacts. Evaluates precision/recall, timing, and contact map
quality across strategies.

In [1]:
# Config
CHECKPOINT_PATH = "../../outputs/exp4"
PDB_ID = "1QYS"
CONTACT_DISTANCE_CUTOFF = 4.0
MAX_NEW_TOKENS = 3440
N_PREFIX = 10  # fixed prefix size for all comparisons
DEVICE = "cuda"

# Sampling strategies to compare
# Each entry: (label, kwargs passed to model.generate)
STRATEGIES = [
    ("Greedy", dict(do_sample=False)),
    ("Beam-2", dict(do_sample=False, num_beams=2)),
    ("Beam-4", dict(do_sample=False, num_beams=4)),
    ("Beam-8", dict(do_sample=False, num_beams=8)),
    ("Temp=0.5", dict(do_sample=True, temperature=0.5, top_k=0)),
    ("Temp=0.8", dict(do_sample=True, temperature=0.8, top_k=0)),
    ("Temp=1.0", dict(do_sample=True, temperature=1.0, top_k=0)),
    ("Temp=1.5", dict(do_sample=True, temperature=1.5, top_k=0)),
    ("Top-k=50", dict(do_sample=True, temperature=1.0, top_k=50)),
    ("Top-k=200", dict(do_sample=True, temperature=1.0, top_k=200)),
    ("Top-p=0.9", dict(do_sample=True, temperature=1.0, top_k=0, top_p=0.9)),
    ("Top-p=0.95", dict(do_sample=True, temperature=1.0, top_k=0, top_p=0.95)),
]

# Number of rollouts for stochastic strategies (beam/greedy are deterministic)
N_ROLLOUTS = 5

In [2]:
# Download & parse PDB structure
import tempfile
import numpy as np
from biotite.database import rcsb
from biotite.structure.io import pdbx
from biotite.structure import filter_amino_acids

path = rcsb.fetch(PDB_ID, "cif", tempfile.gettempdir())
cif = pdbx.CIFFile.read(path)
atoms = pdbx.get_structure(cif.block, model=1)

first_chain = atoms.chain_id[0]
chain_atoms = atoms[(atoms.chain_id == first_chain) & filter_amino_acids(atoms) & (atoms.element != "H")]

res_ids = chain_atoms.res_id
unique_res = sorted(set(res_ids))
res_id_to_pos = {rid: i + 1 for i, rid in enumerate(unique_res)}
sequence = [chain_atoms[chain_atoms.res_id == rid].res_name[0] for rid in unique_res]
seq_len = len(sequence)
print(f"Protein {PDB_ID}: {seq_len} residues, chain {first_chain}")

Protein 1QYS: 92 residues, chain A


In [3]:
# Compute ground-truth contacts
from scipy.spatial import KDTree
from experiments.exp4_contact_prediction.src.data import VALID_ATOMS

coords = chain_atoms.coord
atom_names = chain_atoms.atom_name
atom_res_ids = chain_atoms.res_id

all_known_atoms = set()
for aa in VALID_ATOMS:
    all_known_atoms.update(VALID_ATOMS[aa])

tree = KDTree(coords)
pairs = tree.query_pairs(r=CONTACT_DISTANCE_CUTOFF)

gt_contacts = []
for i, j in pairs:
    ri, rj = atom_res_ids[i], atom_res_ids[j]
    pi, pj = res_id_to_pos.get(ri), res_id_to_pos.get(rj)
    if pi is None or pj is None:
        continue
    if abs(pi - pj) < 2:
        continue
    ai, aj = str(atom_names[i]), str(atom_names[j])
    if ai not in all_known_atoms or aj not in all_known_atoms:
        continue
    aa_i, aa_j = sequence[pi - 1], sequence[pj - 1]
    if aa_i not in VALID_ATOMS or ai not in VALID_ATOMS[aa_i]:
        continue
    if aa_j not in VALID_ATOMS or aj not in VALID_ATOMS[aa_j]:
        continue
    if pi > pj:
        gt_contacts.append((pj, pi, aj, ai))
    else:
        gt_contacts.append((pi, pj, ai, aj))

gt_contacts = sorted(set(gt_contacts), key=lambda c: (-abs(c[0] - c[1]), c[0], c[1], c[2], c[3]))

# Build prompt with prefix
seq_tokens = " ".join(f"<{aa}>" for aa in sequence)
base_prompt = f"<deterministic-positives-only> <begin_sequence> {seq_tokens} <begin_contacts>"
if N_PREFIX > 0:
    prefix_toks = []
    for p1, p2, a1, a2 in gt_contacts[:N_PREFIX]:
        prefix_toks.extend([f"<p{p1}>", f"<p{p2}>", f"<{a1}>", f"<{a2}>"])
    prompt = base_prompt + " " + " ".join(prefix_toks)
else:
    prompt = base_prompt

print(f"Ground-truth contacts: {len(gt_contacts)}")
print(f"Prompt tokens: {len(prompt.split())} (prefix={N_PREFIX})")

/home/ubuntu/llm-protein-experiments/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ground-truth contacts: 860
Prompt tokens: 135 (prefix=10)


In [4]:
# Load model & tokenizer
import torch
from pathlib import Path
from transformers import LlamaForCausalLM
from experiments.exp4_contact_prediction.src.train import create_tokenizer, parse_generated_contacts

ckpt_base = Path(CHECKPOINT_PATH)
ckpt_dirs = sorted(ckpt_base.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
ckpt_path = ckpt_dirs[-1] if ckpt_dirs else ckpt_base
print(f"Loading checkpoint: {ckpt_path}")

tokenizer = create_tokenizer()
model = LlamaForCausalLM.from_pretrained(str(ckpt_path), torch_dtype=torch.bfloat16)
model = model.to(DEVICE).eval()
end_token_id = tokenizer.convert_tokens_to_ids("<end>")
print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

Loading checkpoint: ../../outputs/exp4/checkpoint-31500


Loading weights: 100%|██████████| 147/147 [00:00<00:00, 379.14it/s, Materializing param=model.norm.weight]                              


Model loaded: 984,475,648 parameters


In [5]:
# Helper functions
import time

def contacts_to_matrix(contacts, seq_len):
    matrix = np.zeros((seq_len, seq_len), dtype=np.float32)
    for p1, p2, a1, a2 in contacts:
        if 1 <= p1 <= seq_len and 1 <= p2 <= seq_len:
            matrix[p1 - 1, p2 - 1] = 1
            matrix[p2 - 1, p1 - 1] = 1
    return matrix

def contacts_to_pair_set(contacts):
    return {(min(c[0], c[1]), max(c[0], c[1])) for c in contacts}

def run_generation(prompt, **gen_kwargs):
    """Run generation and return (contacts, valid_grammar, elapsed_seconds)."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=8192)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    kwargs = dict(
        max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=end_token_id,
    )
    kwargs.update(gen_kwargs)

    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(**inputs, **kwargs)
    elapsed = time.time() - t0

    gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
    gen_text = tokenizer.decode(gen_ids, skip_special_tokens=False)
    contacts, valid = parse_generated_contacts(gen_text.split())
    return contacts, valid, elapsed

def compute_accuracy(contacts, gt_pair_set):
    gen_pairs = contacts_to_pair_set(contacts)
    n_gen = len(gen_pairs)
    n_correct = len(gen_pairs & gt_pair_set)
    return {
        "n_gen_pairs": n_gen,
        "n_correct": n_correct,
        "precision": n_correct / n_gen if n_gen > 0 else 0.0,
        "recall": n_correct / len(gt_pair_set) if gt_pair_set else 0.0,
    }

gt_pair_set = contacts_to_pair_set(gt_contacts)
gt_matrix = contacts_to_matrix(gt_contacts, seq_len)
print(f"Ground truth: {len(gt_pair_set)} unique residue pairs")

Ground truth: 250 unique residue pairs


In [ ]:
# Run all strategies
results = {}  # label -> {contacts, accuracy, timings, rollout_contacts, rollout_freq_matrix, ...}

for label, gen_kwargs in STRATEGIES:
    is_deterministic = not gen_kwargs.get("do_sample", False)
    n_runs = 1 if is_deterministic else N_ROLLOUTS

    all_contacts = []
    all_times = []
    all_valid = []

    for r in range(n_runs):
        contacts, valid, elapsed = run_generation(prompt, **gen_kwargs)
        # Prepend prefix contacts
        if N_PREFIX > 0:
            contacts = list(gt_contacts[:N_PREFIX]) + list(contacts)
        all_contacts.append(contacts)
        all_times.append(elapsed)
        all_valid.append(valid)

    # Use first run as the "primary" result (or best for beam search)
    primary_contacts = all_contacts[0]
    gen_only = primary_contacts[N_PREFIX:]
    acc = compute_accuracy(gen_only, gt_pair_set)

    # Build frequency matrix from all rollouts (generated-only portion)
    freq = np.zeros((seq_len, seq_len), dtype=np.float32)
    for contacts in all_contacts:
        gen_only_r = contacts[N_PREFIX:]
        freq += contacts_to_matrix(gen_only_r, seq_len)
    freq /= len(all_contacts)

    # Consensus accuracy (>50%)
    consensus_pairs = set()
    for i in range(seq_len):
        for j in range(i + 1, seq_len):
            if freq[i, j] > 0.5:
                consensus_pairs.add((i + 1, j + 1))
    n_cons = len(consensus_pairs)
    n_cons_correct = len(consensus_pairs & gt_pair_set)

    results[label] = {
        "gen_kwargs": gen_kwargs,
        "primary_contacts": primary_contacts,
        "primary_accuracy": acc,
        "freq_matrix": freq,
        "consensus_precision": n_cons_correct / n_cons if n_cons > 0 else 0.0,
        "consensus_recall": n_cons_correct / len(gt_pair_set) if gt_pair_set else 0.0,
        "consensus_n": n_cons,
        "timings": all_times,
        "n_runs": n_runs,
        "valid_grammar": all_valid,
    }

    avg_t = np.mean(all_times)
    valid_pct = 100 * sum(all_valid) / len(all_valid)
    print(f"{label:12s}: prec={acc['precision']:.1%} rec={acc['recall']:.1%} "
          f"| {acc['n_gen_pairs']} pairs | {n_runs}x {avg_t:.1f}s | valid={valid_pct:.0f}%")

Greedy      : prec=56.7% rec=52.4% | 231 pairs | 1x 19.5s | valid=100%
Beam-2      : prec=48.7% rec=22.8% | 117 pairs | 1x 10.8s | valid=100%
Beam-4      : prec=35.0% rec=14.0% | 100 pairs | 1x 9.2s | valid=100%
Beam-8      : prec=33.7% rec=13.6% | 101 pairs | 1x 9.6s | valid=100%


In [ ]:
# Summary table
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

rows = []
for label, r in results.items():
    acc = r["primary_accuracy"]
    rows.append({
        "Strategy": label,
        "Precision": acc["precision"],
        "Recall": acc["recall"],
        "F1": 2 * acc["precision"] * acc["recall"] / (acc["precision"] + acc["recall"])
             if (acc["precision"] + acc["recall"]) > 0 else 0.0,
        "Pairs": acc["n_gen_pairs"],
        "Cons. Prec": r["consensus_precision"],
        "Cons. Rec": r["consensus_recall"],
        "Cons. Pairs": r["consensus_n"],
        "Avg Time (s)": np.mean(r["timings"]),
        "Valid %": 100 * sum(r["valid_grammar"]) / len(r["valid_grammar"]),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False, float_format="%.3f"))

In [ ]:
# Plot precision vs recall scatter (primary result per strategy)
fig, ax = plt.subplots(figsize=(8, 6))

for _, row in df.iterrows():
    ax.scatter(row["Recall"], row["Precision"], s=80, zorder=3)
    ax.annotate(row["Strategy"], (row["Recall"], row["Precision"]),
                textcoords="offset points", xytext=(6, 4), fontsize=8)

# F1 iso-lines
for f1 in [0.2, 0.3, 0.4, 0.5, 0.6]:
    r_vals = np.linspace(0.01, 1.0, 200)
    p_vals = f1 * r_vals / (2 * r_vals - f1)
    valid = (p_vals > 0) & (p_vals <= 1)
    ax.plot(r_vals[valid], p_vals[valid], "--", color="grey", alpha=0.3, linewidth=0.8)
    # Label the iso-line
    idx = np.argmin(np.abs(p_vals - r_vals))  # where P=R
    if valid[idx]:
        ax.text(r_vals[idx], p_vals[idx], f"F1={f1}", fontsize=7, color="grey", alpha=0.6)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title(f"Precision vs Recall: {PDB_ID} (prefix={N_PREFIX})")
ax.set_xlim(0, None)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Plot timing comparison
fig, ax = plt.subplots(figsize=(10, 4))

labels = list(results.keys())
means = [np.mean(results[l]["timings"]) for l in labels]
stds = [np.std(results[l]["timings"]) if len(results[l]["timings"]) > 1 else 0 for l in labels]

colors = ["#333333" if not results[l]["gen_kwargs"].get("do_sample", False) else "#E57373"
          for l in labels]

bars = ax.bar(range(len(labels)), means, yerr=stds, capsize=3, color=colors)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=35, ha="right")
ax.set_ylabel("Time (seconds)")
ax.set_title(f"Generation Time by Strategy: {PDB_ID} (prefix={N_PREFIX})")

for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{bar.get_height():.1f}s", ha="center", va="bottom", fontsize=7)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="#333333", label="Deterministic"),
    Patch(facecolor="#E57373", label="Stochastic (mean\u00b1std)"),
], fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Contact map heatmaps: ground truth + primary result per strategy
from matplotlib.patches import Circle
from matplotlib.colors import ListedColormap

n_cols = 1 + len(STRATEGIES)
fig, axes = plt.subplots(1, n_cols, figsize=(2.5 * n_cols, 3.2), squeeze=False)
axes = axes[0]
cmap_bw = ListedColormap(["white", "black"])

# Ground truth
axes[0].imshow(gt_matrix, cmap=cmap_bw, vmin=0, vmax=1, origin="upper", aspect="equal")
axes[0].set_title(f"Ground Truth\n{len(gt_pair_set)} pairs", fontsize=8)
axes[0].set_xlabel("Residue", fontsize=7)
axes[0].set_ylabel("Residue", fontsize=7)

for i, (label, _) in enumerate(STRATEGIES):
    ax = axes[i + 1]
    r = results[label]
    gen_only = r["primary_contacts"][N_PREFIX:]
    m = contacts_to_matrix(gen_only, seq_len)
    ax.imshow(m, cmap=cmap_bw, vmin=0, vmax=1, origin="upper", aspect="equal")

    # Red circles for prefix
    if N_PREFIX > 0:
        prefix_pairs_plotted = set()
        for p1, p2, a1, a2 in gt_contacts[:N_PREFIX]:
            pair = (min(p1, p2) - 1, max(p1, p2) - 1)
            if pair not in prefix_pairs_plotted:
                prefix_pairs_plotted.add(pair)
                for (row, col) in [(pair[0], pair[1]), (pair[1], pair[0])]:
                    ax.add_patch(Circle((col, row), radius=1.5, fill=False,
                                        edgecolor="red", linewidth=1.0))

    acc = r["primary_accuracy"]
    ax.set_title(f"{label}\nP={acc['precision']:.0%} R={acc['recall']:.0%}", fontsize=8)
    ax.set_xlabel("Residue", fontsize=7)
    ax.tick_params(labelsize=6)

fig.suptitle(f"Contact Maps by Strategy: {PDB_ID} (prefix={N_PREFIX})", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Rollout frequency heatmaps for stochastic strategies
stochastic = [(l, r) for l, r in results.items() if r["n_runs"] > 1]

if stochastic:
    n_cols = 1 + len(stochastic)
    fig, axes = plt.subplots(1, n_cols, figsize=(2.5 * n_cols, 3.5), squeeze=False)
    axes = axes[0]

    axes[0].imshow(gt_matrix, cmap="Greys", vmin=0, vmax=1, origin="upper", aspect="equal")
    axes[0].set_title(f"Ground Truth\n{len(gt_pair_set)} pairs", fontsize=8)
    axes[0].set_xlabel("Residue", fontsize=7)
    axes[0].set_ylabel("Residue", fontsize=7)

    for i, (label, r) in enumerate(stochastic):
        ax = axes[i + 1]
        im = ax.imshow(r["freq_matrix"], cmap="hot_r", vmin=0, vmax=1,
                       origin="upper", aspect="equal")
        cp, cr = r["consensus_precision"], r["consensus_recall"]
        ax.set_title(f"{label}\nCons: P={cp:.0%} R={cr:.0%}", fontsize=8)
        ax.set_xlabel("Residue", fontsize=7)
        ax.tick_params(labelsize=6)

        # Prefix circles
        if N_PREFIX > 0:
            prefix_pairs_plotted = set()
            for p1, p2, a1, a2 in gt_contacts[:N_PREFIX]:
                pair = (min(p1, p2) - 1, max(p1, p2) - 1)
                if pair not in prefix_pairs_plotted:
                    prefix_pairs_plotted.add(pair)
                    for (row, col) in [(pair[0], pair[1]), (pair[1], pair[0])]:
                        ax.add_patch(Circle((col, row), radius=1.5, fill=False,
                                            edgecolor="blue", linewidth=1.0))

    cbar = fig.colorbar(im, ax=axes.tolist(), shrink=0.7, pad=0.02)
    cbar.set_label(f"Fraction of {N_ROLLOUTS} rollouts", fontsize=8)

    fig.suptitle(f"Rollout Frequency Maps by Strategy: {PDB_ID} (prefix={N_PREFIX})",
                 fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No stochastic strategies to plot.")

In [ ]:
# F1 bar chart comparing all strategies
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Primary F1
ax = axes[0]
colors_f1 = sns.color_palette("husl", len(df))
bars = ax.barh(range(len(df)), df["F1"], color=colors_f1)
ax.set_yticks(range(len(df)))
ax.set_yticklabels(df["Strategy"])
ax.set_xlabel("F1 Score")
ax.set_title("Primary (single run) F1")
ax.set_xlim(0, 1)
for bar, val in zip(bars, df["F1"]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=8)

# Consensus F1 (for stochastic, same as primary for deterministic)
ax = axes[1]
cons_f1 = []
for _, row in df.iterrows():
    cp, cr = row["Cons. Prec"], row["Cons. Rec"]
    f1 = 2 * cp * cr / (cp + cr) if (cp + cr) > 0 else 0.0
    cons_f1.append(f1)
bars = ax.barh(range(len(df)), cons_f1, color=colors_f1)
ax.set_yticks(range(len(df)))
ax.set_yticklabels(df["Strategy"])
ax.set_xlabel("F1 Score")
ax.set_title(f"Consensus (>50% of {N_ROLLOUTS} runs) F1")
ax.set_xlim(0, 1)
for bar, val in zip(bars, cons_f1):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=8)

plt.suptitle(f"F1 Comparison: {PDB_ID} (prefix={N_PREFIX})", fontsize=13)
plt.tight_layout()
plt.show()